# Terrain, TINs, and local shape

Ground grids, local PCA features, and TIN queries. These are how you tell pavement from a pole from a curb face, and how you read elevation off a surface that honors breaklines.

You will not be asked to implement constrained Delaunay. You may be asked what a breakline does (triangles are not allowed to cross it, because the curb is a slope break) and how to interpolate Z once the mesh exists.


These cells are PyTorch. Geometry wants float64, so the setup cell calls `torch.set_default_dtype(torch.float64)`. The library default is float32, and a plane SVD or a Kabsch in float32 will miss a hand check that passes in float64.


In [ ]:
import torch

torch.set_default_dtype(torch.float64)
torch.set_printoptions(precision=4, sci_mode=False)


## 1. Minimum-Z grid

Bin an `(N, 3)` cloud into a regular XY grid and return the minimum Z in each cell. This is the simple version of a ground grid.

`origin_xy` is the minimum corner. `cell` is the square size. `shape` is `(n_rows, n_cols)`. Row 0 is the **minimum-Y** band, so the row index increases with Y. This is not an image.

Cells are half-open: the lower edge belongs to the cell, the upper edge belongs to the next cell. A point exactly on `x = origin_x + cell` is in the next column. Points outside the grid are ignored. Empty cells are NaN.

The cell value has to be the minimum even when a higher point is written after a lower one. Ordinary fancy indexing keeps one write and depends on order.

Hand check: origin `(0,0)`, cell `1`, shape `(2, 3)`.

- `(0.1, 0.1, 2)` and later `(0.9, 0.2, 5)` → cell `(row 0, col 0)` is 2, not 5
- `(1.2, 0.2, 4)` and `(1.0, 0.2, 1)` → cell `(0, 1)` is 1, because `x = 1` starts column 1
- `(1.2, 1.5, 8)` → cell `(1, 1)` is 8
- `(5, 5, 1)` and `(-0.2, 0.2, 9)` fall outside

<details><summary>Hint</summary>

Flatten the cell index and call `scatter_reduce_(0, flat, z, reduce='amin', include_self=True)`. Initialize with `+inf` and replace leftovers with NaN. An ordinary index assignment keeps one write and depends on order.

</details>


In [ ]:
import torch

def grid_min_z(points, origin_xy, cell, shape):
    """
    Minimum Z in each cell of a regular XY grid.

    origin_xy is the minimum corner (x, y). `cell` is the square size.
    `shape` is (n_rows, n_cols): row 0 is the minimum-Y band, so the row index
    increases with Y (this is not an image, where row 0 is the top).
    Cells are half-open: the lower edge is included, the upper edge belongs to
    the next cell. Empty cells are NaN.
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_default_dtype(torch.float64)
DTYPE = torch.float64

def _t(x, dtype=DTYPE):
    if isinstance(x, torch.Tensor):
        return x.to(dtype=dtype)
    return torch.as_tensor(x, dtype=dtype)

def grid_min_z(points, origin_xy, cell, shape):
    """Minimum Z in each cell. Row 0 is the minimum-Y band.

    Empty cells are NaN. scatter_reduce_(reduce='amin') keeps the minimum
    when several points land in one cell.
    """
    points = _t(points)
    origin_xy = _t(origin_xy)
    n_rows, n_cols = shape
    ij = torch.floor((points[:, :2] - origin_xy) / cell).to(torch.int64)
    cols = ij[:, 0]
    rows = ij[:, 1]
    inside = (rows >= 0) & (rows < n_rows) & (cols >= 0) & (cols < n_cols)
    acc = torch.full((n_rows * n_cols,), torch.inf, dtype=points.dtype)
    flat = rows[inside] * n_cols + cols[inside]
    acc.scatter_reduce_(0, flat, points[inside, 2], reduce="amin", include_self=True)
    grid = acc.view(n_rows, n_cols)
    grid = torch.where(torch.isinf(grid), torch.full_like(grid, torch.nan), grid)
    return grid

def check_grid_min_z():
    pts = torch.tensor(
        [
            [0.1, 0.1, 2.0],
            [0.9, 0.2, 5.0],
            [1.2, 0.2, 4.0],
            [1.2, 1.5, 8.0],
            [1.0, 0.2, 1.0],
            [5.0, 5.0, 1.0],
            [-0.2, 0.2, 9.0],
        ]
    )
    grid = grid_min_z(pts, origin_xy=(0.0, 0.0), cell=1.0, shape=(2, 3))
    assert torch.allclose(grid[0, 0], torch.tensor(2.0))
    assert torch.allclose(grid[0, 1], torch.tensor(1.0))
    assert torch.allclose(grid[1, 1], torch.tensor(8.0))
    assert torch.isnan(grid[0, 2]) and torch.isnan(grid[1, 0]) and torch.isnan(grid[1, 2])
    print("passed")

check_grid_min_z()


## 2. Linearity, planarity, sphericity

You are given one neighborhood, already gathered, as `(N, 3)`. In production those neighbors come from a KD-tree.

Compute the population covariance (divide by `N`). Take its eigenvalues `λ_max ≥ λ_mid ≥ λ_min` and return

- linearity `(λ_max - λ_mid) / λ_max`
- planarity `(λ_mid - λ_min) / λ_max`
- sphericity `λ_min / λ_max`
- the principal direction (largest eigenvector)
- the normal (smallest eigenvector)

`torch.linalg.eigh` returns eigenvalues in **ascending** order, and eigenvector signs are arbitrary. Flip each vector so its largest component is positive. If the neighborhood is degenerate (fewer than 3 points, or no spread), return zeros and any unit direction/normal you document.

How to talk about the numbers:

- a pole is linear, and `|direction · up|` is near 1
- ground is planar, and `|normal · up|` is near 1
- a curb face is planar, and `|normal · up|` is near 0

Hand check: four points on the Z axis have linearity 1 and direction `(0,0,1)`. The unit square in the XY plane has planarity 1 and a vertical normal. The unit square in the YZ plane (`x = 0`) has planarity 1 and a horizontal normal.


In [ ]:
import torch

def shape_features(points):
    """
    Local covariance features of one neighborhood.

    Returns linearity, planarity, sphericity, principal direction, normal.
    Eigenvalues are ordered λ_max ≥ λ_mid ≥ λ_min from the population
    covariance (divided by N, not N-1; the ratios are the same either way):

        linearity  = (λ_max - λ_mid) / λ_max
        planarity  = (λ_mid - λ_min) / λ_max
        sphericity = λ_min / λ_max

    Direction is the largest-variance axis. Normal is the smallest. A pole has
    high linearity and |direction · up| near 1. Ground has high planarity and
    |normal · up| near 1. A curb face has high planarity and |normal · up|
    near 0. In production the neighborhood comes from a KD-tree.
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_default_dtype(torch.float64)
DTYPE = torch.float64

def _t(x, dtype=DTYPE):
    if isinstance(x, torch.Tensor):
        return x.to(dtype=dtype)
    return torch.as_tensor(x, dtype=dtype)

def shape_features(points):
    """Linearity, planarity, sphericity, principal direction, normal.

    Eigenvalues from the population covariance, largest to smallest:
    L = (l1-l2)/l1, P = (l2-l3)/l1, S = l3/l1.
    torch.linalg.eigh returns eigenvalues ascending.
    """
    points = _t(points)
    direction = torch.tensor([1.0, 0.0, 0.0], dtype=points.dtype)
    normal = torch.tensor([0.0, 0.0, 1.0], dtype=points.dtype)
    if points.shape[0] < 3:
        return 0.0, 0.0, 0.0, direction, normal
    centered = points - points.mean(dim=0)
    cov = (centered.T @ centered) / points.shape[0]
    evals, evecs = torch.linalg.eigh(cov)
    l_min, l_mid, l_max = evals
    if l_max < 1e-15:
        return 0.0, 0.0, 0.0, direction, normal
    linearity = float((l_max - l_mid) / l_max)
    planarity = float((l_mid - l_min) / l_max)
    sphericity = float(l_min / l_max)
    direction = evecs[:, -1].clone()
    normal = evecs[:, 0].clone()
    for vec in (direction, normal):
        k = int(torch.argmax(vec.abs()))
        if vec[k] < 0.0:
            vec *= -1.0
    return linearity, planarity, sphericity, direction, normal

def check_shape_features():
    line = torch.tensor([[0.0, 0.0, 0.0], [0.0, 0.0, 1.0], [0.0, 0.0, 2.0], [0.0, 0.0, 3.0]])
    linearity, planarity, sphericity, direction, _ = shape_features(line)
    assert linearity > 0.99
    assert planarity < 1e-8
    assert sphericity < 1e-8
    assert torch.allclose(direction, torch.tensor([0.0, 0.0, 1.0]), atol=1e-8)
    plane = torch.tensor([[0.0, 0.0, 0.0], [1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [1.0, 1.0, 0.0]])
    linearity, planarity, sphericity, _, normal = shape_features(plane)
    assert linearity < 1e-8
    assert planarity > 0.99
    assert sphericity < 1e-8
    assert torch.allclose(normal.abs(), torch.tensor([0.0, 0.0, 1.0]), atol=1e-8)
    wall = torch.tensor([[0.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0], [0.0, 1.0, 1.0]])
    _, planarity, _, _, normal = shape_features(wall)
    assert planarity > 0.99
    assert torch.allclose(normal.abs(), torch.tensor([1.0, 0.0, 0.0]), atol=1e-8)
    print("passed")

check_shape_features()


## 3. Interpolate a TIN

`vertices` is `(V, 3)`, `faces` is `(F, 3)` integer indices, `queries` is `(N, 2)` XY. Return Z for each query, and NaN when no triangle contains it.

Use barycentric coordinates in XY, then interpolate Z. Treat the boundary as inside. If two triangles contain a query (a shared edge), the first face in `faces` wins. On a continuous mesh the Z values agree, so the tie does not matter. A degenerate triangle with zero XY area contains nothing.

Hand check:

```
vertices (0,0,0), (2,0,0), (0,2,10), (2,2,6)
faces (0, 1, 2) and (1, 3, 2)
```

- `(0.5, 0.5)` → `2.5`
- `(1.5, 1.5)` → `5.5`
- `(1, 1)` on the shared edge → `5`
- `(3, 3)` → NaN

<details><summary>Hint</summary>

For triangle `a, b, c` and query `p`: `v0 = b - a`, `v1 = c - a`, `v2 = p - a`. Let `dij = vi · vj`. Then `v = (d11 d20 - d01 d21) / denom`, `w = (d00 d21 - d01 d20) / denom`, `u = 1 - v - w`, with `denom = d00 d11 - d01²`. Broadcast queries against faces so the distance array is `(N, F)`.

</details>


In [ ]:
import torch

def interpolate_tin(vertices, faces, queries):
    """
    Interpolate Z from a 2.5D triangle mesh. NaN when a query misses every triangle.

    vertices: (V, 3), faces: (F, 3) indices, queries: (N, 2) XY.
    Barycentric coordinates are computed in XY. The first containing face wins.
    Shared edges agree when the mesh is continuous, so the tie does not change Z.
    A breakline is a constraint on how the mesh was built: triangles are not
    allowed to cross it. Querying the mesh does not rebuild that triangulation.
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_default_dtype(torch.float64)
DTYPE = torch.float64

def _t(x, dtype=DTYPE):
    if isinstance(x, torch.Tensor):
        return x.to(dtype=dtype)
    return torch.as_tensor(x, dtype=dtype)

def interpolate_tin(vertices, faces, queries):
    """Interpolate Z from a 2.5D mesh. NaN outside every triangle."""
    vertices = _t(vertices)
    faces = torch.as_tensor(faces, dtype=torch.long)
    queries = _t(queries)
    tri = vertices[faces]
    a = tri[:, 0, :2]
    b = tri[:, 1, :2]
    c = tri[:, 2, :2]
    v0 = b - a
    v1 = c - a
    v2 = queries[:, None, :] - a[None, :, :]
    d00 = (v0 * v0).sum(dim=1)
    d01 = (v0 * v1).sum(dim=1)
    d11 = (v1 * v1).sum(dim=1)
    d20 = (v2 * v0[None, :, :]).sum(dim=-1)
    d21 = (v2 * v1[None, :, :]).sum(dim=-1)
    denom = d00 * d11 - d01 * d01
    inv = torch.zeros_like(denom)
    ok = denom.abs() > 1e-12
    inv = torch.where(ok, 1.0 / denom, inv)
    v = (d11 * d20 - d01 * d21) * inv
    w = (d00 * d21 - d01 * d20) * inv
    u = 1.0 - v - w
    eps = 1e-8
    inside = ok & (u >= -eps) & (v >= -eps) & (w >= -eps)
    z = u * tri[None, :, 0, 2] + v * tri[None, :, 1, 2] + w * tri[None, :, 2, 2]
    any_inside = inside.any(dim=1)
    idx = inside.to(torch.int64).argmax(dim=1)
    out = z[torch.arange(queries.shape[0]), idx]
    return torch.where(any_inside, out, torch.full_like(out, torch.nan))

def check_interpolate_tin():
    vertices = torch.tensor(
        [
            [0.0, 0.0, 0.0],
            [2.0, 0.0, 0.0],
            [0.0, 2.0, 10.0],
            [2.0, 2.0, 6.0],
        ]
    )
    faces = torch.tensor([[0, 1, 2], [1, 3, 2]])
    queries = torch.tensor([[0.5, 0.5], [1.5, 1.5], [1.0, 1.0], [3.0, 3.0]])
    z = interpolate_tin(vertices, faces, queries)
    assert torch.allclose(z[:3], torch.tensor([2.5, 5.5, 5.0]))
    assert torch.isnan(z[3])
    print("passed")

check_interpolate_tin()


## 4. Drop long triangles

A TIN will bridge gaps with long, thin triangles unless you remove them. Drop every face that has any edge longer than `max_edge`. Return the surviving faces, in their original order.

Hand check: vertices `(0,0,0), (1,0,0), (0,1,0), (10,0,0)` and faces `(0,1,2), (0,1,3)`. With `max_edge = 2`, only the first face remains. The second face has edges of length 9 and 10.


In [ ]:
import torch

def drop_long_triangles(vertices, faces, max_edge):
    """
    Drop faces that have any edge longer than max_edge.

    This is the sliver / gap-span filter used when a TIN bridges empty space.
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_default_dtype(torch.float64)
DTYPE = torch.float64

def _t(x, dtype=DTYPE):
    if isinstance(x, torch.Tensor):
        return x.to(dtype=dtype)
    return torch.as_tensor(x, dtype=dtype)

def drop_long_triangles(vertices, faces, max_edge):
    """Drop faces with any edge longer than max_edge."""
    vertices = _t(vertices)
    faces = torch.as_tensor(faces, dtype=torch.long)
    tri = vertices[faces]
    e01 = torch.linalg.norm(tri[:, 0] - tri[:, 1], dim=1)
    e12 = torch.linalg.norm(tri[:, 1] - tri[:, 2], dim=1)
    e20 = torch.linalg.norm(tri[:, 2] - tri[:, 0], dim=1)
    keep = (e01 <= max_edge) & (e12 <= max_edge) & (e20 <= max_edge)
    return faces[keep]

def check_drop_long_triangles():
    vertices = torch.tensor([[0.0, 0.0, 0.0], [1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [10.0, 0.0, 0.0]])
    faces = torch.tensor([[0, 1, 2], [0, 1, 3]])
    kept = drop_long_triangles(vertices, faces, max_edge=2.0)
    assert torch.equal(kept, faces[:1])
    print("passed")

check_drop_long_triangles()


## 5. Fit a circle to a pole

A vertical pole, seen in plan view, is a circle in XY. Fit its center and radius with the Kåsa algebraic fit, solved by `torch.linalg.lstsq(...).solution` (not by forming `(AᵀA)⁻¹` yourself).

Expand `(x - cx)² + (y - cy)² = r²` into `x² + y² = 2 cx x + 2 cy y + c`, then `r² = c + cx² + cy²`.

Hand check: 40 points on the circle centered at `(3, -1)` with radius 2. You should recover that center and radius.

Say the limitation out loud. Points that lie exactly on a short arc still recover the true circle, because they satisfy the equation. The centroid of that arc does not: it sits on the visible side. With a centimeter of noise on a short arc, the algebraic fit moves too. A lidar pole is a partial arc, often seen from one side, so "the center is the mean of the XY points" is the wrong tool, and a one-shot algebraic fit on a thin noisy arc is fragile. RANSAC, or a geometric fit, is the production version.

<details><summary>Hint</summary>

`A = [2x, 2y, 1]`, `b = x² + y²`, `cx, cy, c = torch.linalg.lstsq(A, b).solution`. Reject a non-positive `r²`.

</details>


In [ ]:
import torch

def fit_circle_kasa(xy):
    """
    Algebraic circle fit (Kåsa) via linear least squares.

    xy: (N, 2). Returns center (2,) and radius.
    Minimizing the algebraic residual is not the same as minimizing geometric
    distance. A full ring is fine. A short lidar arc of a pole is biased,
    because the scanner only sees one side and this fit pulls the center
    toward the visible points. Say that limitation out loud.
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_default_dtype(torch.float64)
DTYPE = torch.float64

def _t(x, dtype=DTYPE):
    if isinstance(x, torch.Tensor):
        return x.to(dtype=dtype)
    return torch.as_tensor(x, dtype=dtype)

def fit_circle_kasa(xy):
    """Algebraic circle fit. Returns center (2,) and radius."""
    xy = _t(xy)
    if xy.shape[0] < 3:
        raise ValueError("need at least 3 points")
    A = torch.column_stack((2.0 * xy[:, 0], 2.0 * xy[:, 1], torch.ones(xy.shape[0], dtype=xy.dtype)))
    b = (xy * xy).sum(dim=1)
    sol = torch.linalg.lstsq(A, b).solution
    cx, cy, c = sol
    radius_sq = c + cx * cx + cy * cy
    if radius_sq <= 0:
        raise ValueError("circle fit produced a non-positive radius")
    return torch.stack((cx, cy)), torch.sqrt(radius_sq)

def check_fit_circle_kasa():
    theta = torch.linspace(0.0, 2.0 * torch.pi, 40 + 1)[:-1]
    xy = torch.stack((3.0 + 2.0 * torch.cos(theta), -1.0 + 2.0 * torch.sin(theta)), dim=1)
    center, radius = fit_circle_kasa(xy)
    assert torch.allclose(center, torch.tensor([3.0, -1.0]), atol=1e-8)
    assert torch.allclose(radius, torch.tensor(2.0), atol=1e-8)
    print("passed")

check_fit_circle_kasa()
